# ARA-PPO v3 — Regime-Aware Ensemble + Constraint-Aware PPO
## Hybrid Architecture for Commodity Market Optimization

Two-stage pipeline:

**Stage 1 — Forecasting (supervised)**
Five base forecasters predict next-day log-return (μ) and volatility (σ):

| Forecaster | Inductive bias |
|---|---|
| LSTM | temporal momentum, autoregressive sequences |
| Transformer | long-range dependencies via self-attention |
| XGBoost | tabular / non-linear feature interactions |
| CNN | local pattern extraction |
| GARCH(1,1) | conditional volatility |

**Stage 2 — Regime-aware ensemble**
An HMM (4 states, fit per split) outputs regime probabilities. A learned per-regime weight matrix combines forecaster predictions: 
$$\mu_{ens} = \sum_i w_i(\text{regime}) \cdot \mu_i, \quad \sigma_{ens}^2 = \sum_i w_i \sigma_i^2 + \sum_i w_i (\mu_i - \mu_{ens})^2$$

**Stage 3 — Constraint-Aware PPO**
A compact policy receives the 12-dim state $[\mu_{ens}, \sigma_{ens}, \text{Sharpe}, \text{regime}_{1..4}, \text{portfolio}_{1..5}]$ and outputs a position bounded by the Kelly fraction $|μ/σ²|$.

Why this works where pure PPO fails:
- Forecasting is solved by supervised learning (sample-efficient)
- PPO only handles position sizing on a 12-dim state (small + tractable)
- Constraint prevents over-leverage on noisy forecasts

Walk-forward design: per split, fit forecasters → fit HMM → fit ensemble → train PPO → evaluate. No lookahead.

In [ ]:
import sys, os, json, warnings, time, io, contextlib
import numpy as np
import pandas as pd
import torch as th
import yaml
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
warnings.filterwarnings('ignore')

def _find_project_root(start):
    candidate = os.path.abspath(start)
    for _ in range(4):
        if (os.path.isdir(os.path.join(candidate, 'src')) and
                os.path.isdir(os.path.join(candidate, 'data'))):
            return candidate
        candidate = os.path.dirname(candidate)
    raise RuntimeError(f'Cannot locate project root from {start!r}.')

PROJECT_ROOT = _find_project_root(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.environment    import CommodityTradingEnv, WalkForwardEnvFactory
from src.regime_hmm     import RegimeHMM
from src.forecasters    import build_default_forecasters
from src.ensemble       import (RegimeAwareEnsemble, stack_forecaster_predictions,
                                 kelly_per_regime_from_hmm, classify_regimes)
from src.constraint_ppo import HybridForecastEnv, make_compact_ppo

print('Project root:', PROJECT_ROOT)
print('torch:', th.__version__, ' device:', 'cuda' if th.cuda.is_available() else 'cpu')

In [ ]:
DATA_DIR    = os.path.join(PROJECT_ROOT, 'data')
CONFIGS_DIR = os.path.join(PROJECT_ROOT, 'configs')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

df = pd.read_parquet(os.path.join(DATA_DIR, 'features', 'WTI_Crude_Oil_features.parquet'))
with open(os.path.join(DATA_DIR, 'features', 'feature_config.json')) as f:
    feat_cfg = json.load(f)
FEATURES = feat_cfg['all_features']

with open(os.path.join(CONFIGS_DIR, 'best_env_config.yaml')) as f:
    ENV_CONFIG = yaml.safe_load(f)

with open(os.path.join(DATA_DIR, 'features', 'walkforward_splits.json')) as f:
    raw_splits = json.load(f)
SPLITS = [
    {k: pd.Timestamp(v) for k, v in s.items()} for s in raw_splits
]

print(f'Features: {len(FEATURES)}  |  Splits: {len(SPLITS)}')
print(f'First split: {SPLITS[0]["train_start"].date()} -> {SPLITS[0]["test_end"].date()}')
print(f'Last  split: {SPLITS[-1]["train_start"].date()} -> {SPLITS[-1]["test_end"].date()}')

In [ ]:
# v3 hybrid hyperparameters

# --- Forecaster training ---
FORECASTER_LOOKBACK = 60
FORECASTER_EPOCHS   = 5       # was 15: caused overfit on early splits with limited data

# --- HMM regime detector ---
HMM_STATES     = 3       # v3.2: dropped from 4 (regime 3 was always uniform/dead)
HMM_VOL_WINDOW = 20
HMM_SEED       = 42

# --- PPO ---
PPO_TIMESTEPS_BASE = 200_000   # smaller policy, can afford fewer steps
PPO_LR             = 3e-4
PPO_HIDDEN         = 64
PPO_KELLY_FRAC     = 0.5       # half-Kelly fallback (used only if kelly_per_regime is None)
PPO_GAMMA          = 0.97
PPO_GAE_LAMBDA     = 0.95
PPO_CLIP           = 0.2
PPO_ENT_COEF       = 0.01

# --- v3.1 risk-aware features ---
USE_SIGMA_FLOOR    = True      # sigma_eff = max(sigma_ens, sigma_garch, sigma_train_std)
USE_REGIME_KELLY   = True      # Kelly fraction varies per HMM regime
USE_VOL_TARGETING  = True      # Scale position by target_vol / realized_vol
TARGET_ANN_VOL     = 0.08      # v3.3 lowered (was 0.15) — match forecaster skill
VOL_SCALER_MIN     = 0.3       # never shrink below 30% of policy output
VOL_SCALER_MAX     = 1.5       # never amplify above 150%
GARCH_FORECASTER_IDX = 4       # GARCH is the 5th forecaster
KELLY_BASE         = 0.5       # base for kelly_per_regime_from_hmm()

# --- v3.2 OOD + forecast-quality feedback ---
USE_OOD_ATTENUATION   = True
OOD_THRESHOLD         = 0.02   # legacy fallback (used when ood_p90 unset)
OOD_ATTENUATION_ALPHA = 0.5
USE_EWMA_CORR         = True
EWMA_WINDOW           = 60     # v3.3 longer window (was 20) for stable correlation

# --- v3.3 new features ---
USE_AUTOCORR_FEATURE      = True   # lag-1 autocorr of returns (chop detector)
AUTOCORR_WINDOW           = 20
USE_DISAGREEMENT_IN_SIGMA = True   # sigma_final includes forecaster disagreement
USE_ROLLING_VOL_IN_SIGMA  = True   # sigma_final includes rolling realised vol
ROLLING_VOL_WINDOW        = 20
DETACH_KELLY              = True   # prevent gradient explosion on high-Kelly days
USE_REGIME_SPECIALISED_TRAINING = True   # weighted forecaster training
ENSEMBLE_BASE_TEMP        = 1.0
ENSEMBLE_ALPHA_TEMP       = 1.5

# --- Walk-forward filter (mirror v2 setup) ---
MIN_TRAIN_DAYS = 756           # skip splits with < 3 years of training data
EVAL_EPISODES  = 5
SEED           = 42

print('Hyperparameters configured.')
print(f'  forecaster epochs : {FORECASTER_EPOCHS}')
print(f'  HMM states        : {HMM_STATES}')
print(f'  PPO timesteps     : {PPO_TIMESTEPS_BASE:,}  (per split)')
print(f'  Kelly fraction    : {PPO_KELLY_FRAC}')
print(f'  Sigma floor       : {USE_SIGMA_FLOOR}')
print(f'  Regime-aware Kelly: {USE_REGIME_KELLY}')
print(f'  Vol targeting     : {USE_VOL_TARGETING}  (target={TARGET_ANN_VOL})')
print(f'  OOD attenuation   : {USE_OOD_ATTENUATION}  (alpha={OOD_ATTENUATION_ALPHA})')
print(f'  EWMA corr feedback: {USE_EWMA_CORR}  (window={EWMA_WINDOW})')
print(f'  Autocorr feature  : {USE_AUTOCORR_FEATURE}  (chop detector)')
print(f'  Disagreement in σ : {USE_DISAGREEMENT_IN_SIGMA}')
print(f'  Rolling vol in σ  : {USE_ROLLING_VOL_IN_SIGMA}')
print(f'  Detach Kelly grad : {DETACH_KELLY}')
print(f'  Regime-specialised forecasters: {USE_REGIME_SPECIALISED_TRAINING}')

## Single-split sanity check
Run the full pipeline on one split before launching the 33-split walk-forward loop. This catches bugs cheaply.

In [ ]:
TEST_SPLIT_IDX = 15  # mid-range split for sanity check

split = SPLITS[TEST_SPLIT_IDX]
train_df_s = df[(df.index >= split['train_start']) & (df.index < split['train_end'])].copy()
test_df_s  = df[(df.index >= split['test_start'])  & (df.index < split['test_end'])].copy()
print(f'Single-split test: split {TEST_SPLIT_IDX+1}  train={len(train_df_s)}d  test={len(test_df_s)}d')

# 1. Train 5 forecasters
t0 = time.time()
forecasters_s = build_default_forecasters(FEATURES, lookback=FORECASTER_LOOKBACK, epochs=3)  # quick
buf = io.StringIO()
# v3.3: HMM must be fit FIRST so we can use regime probs for forecaster sample weights
with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
    hmm_pre = RegimeHMM(n_states=HMM_STATES, vol_window=HMM_VOL_WINDOW, random_state=HMM_SEED).fit(train_df_s)
train_rp_pre = hmm_pre.predict_proba(train_df_s)
primary_regime_s = {'lstm': 0, 'transformer': 0, 'xgboost': 1, 'cnn': 1, 'garch': 0}
buf = io.StringIO()
with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
    for fc in forecasters_s:
        if USE_REGIME_SPECIALISED_TRAINING:
            pri = min(primary_regime_s.get(fc.name, 0), HMM_STATES - 1)
            sw = train_rp_pre[:, pri] + 0.3
            fc.fit(train_df_s, sample_weights=sw)
        else:
            fc.fit(train_df_s)
print(f'  forecasters fit:    {time.time()-t0:5.1f}s')

# 2. Stack forecaster predictions
train_mus_s, train_sigs_s = stack_forecaster_predictions(forecasters_s, train_df_s)
test_mus_s,  test_sigs_s  = stack_forecaster_predictions(forecasters_s, test_df_s)

# 3. Fit HMM
t0 = time.time()
buf = io.StringIO()
with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
    hmm_s = RegimeHMM(n_states=HMM_STATES, vol_window=HMM_VOL_WINDOW, random_state=HMM_SEED).fit(train_df_s)
train_rp_s = hmm_s.predict_proba(train_df_s)
test_rp_s  = hmm_s.predict_proba(test_df_s)
print(f'  HMM fit:            {time.time()-t0:5.1f}s')

# 4. Fit ensemble
t0 = time.time()
target_train_s = np.roll(train_df_s['log_return'].fillna(0).values, -1)
ens_s = RegimeAwareEnsemble(
    n_regimes=HMM_STATES, n_forecasters=5,
    base_temp=ENSEMBLE_BASE_TEMP, alpha_temp=ENSEMBLE_ALPHA_TEMP,
).fit(train_mus_s, train_rp_s, target_train_s)
print(f'  ensemble fit:       {time.time()-t0:5.1f}s')
W = ens_s.regime_weight_matrix()
names = [fc.name for fc in forecasters_s]
print(f'  regime weight matrix:')
header_str = '              ' + ''.join(f'{n[:10]:>11s}' for n in names)
print(header_str)
for r in range(HMM_STATES):
    print(f'    regime {r}: ' + ''.join(f'{W[r,k]:11.3f}' for k in range(5)))

# 5. Build envs
factory = WalkForwardEnvFactory(df, FEATURES, SPLITS, ENV_CONFIG)
buf = io.StringIO()
with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
    base_train_s, base_test_s, _ = factory.get_split(TEST_SPLIT_IDX)
ftrain_mus_s, ftrain_sigs_s = stack_forecaster_predictions(forecasters_s, base_train_s.df)
ftest_mus_s,  ftest_sigs_s  = stack_forecaster_predictions(forecasters_s, base_test_s.df)
ftrain_rp_s = hmm_s.predict_proba(base_train_s.df)
ftest_rp_s  = hmm_s.predict_proba(base_test_s.df)
# v3.1: compute sigma_train_floor and per-regime Kelly
sigma_train_floor_s = float(train_df_s['log_return'].std()) if USE_SIGMA_FLOOR else 0.0
regime_info_s       = classify_regimes(hmm_s)
kelly_per_regime_s  = regime_info_s['kelly_caps'] if USE_REGIME_KELLY else None
target_vol_s        = TARGET_ANN_VOL if USE_VOL_TARGETING else None
ood_p90_s           = float(np.percentile(train_mus_s.std(axis=1), 90)) if USE_OOD_ATTENUATION else None

hybrid_train_s = HybridForecastEnv(
    base_train_s, ftrain_mus_s, ftrain_sigs_s, ftrain_rp_s, ens_s,
    sigma_garch_idx=GARCH_FORECASTER_IDX,
    sigma_train_floor=sigma_train_floor_s,
    target_vol=target_vol_s,
    vol_scaler_min=VOL_SCALER_MIN,
    vol_scaler_max=VOL_SCALER_MAX,
    use_ood_attenuation=USE_OOD_ATTENUATION,
    ood_threshold=OOD_THRESHOLD,
    ood_p90=ood_p90_s,
    ood_attenuation_alpha=OOD_ATTENUATION_ALPHA,
    use_ewma_corr=USE_EWMA_CORR,
    ewma_window=EWMA_WINDOW,
    use_autocorr_feature=USE_AUTOCORR_FEATURE,
    autocorr_window=AUTOCORR_WINDOW,
    use_disagreement_in_sigma=USE_DISAGREEMENT_IN_SIGMA,
    use_rolling_vol_in_sigma=USE_ROLLING_VOL_IN_SIGMA,
    rolling_vol_window=ROLLING_VOL_WINDOW,
)
hybrid_test_s  = HybridForecastEnv(
    base_test_s, ftest_mus_s, ftest_sigs_s, ftest_rp_s, ens_s,
    sigma_garch_idx=GARCH_FORECASTER_IDX,
    sigma_train_floor=sigma_train_floor_s,
    target_vol=target_vol_s,
    vol_scaler_min=VOL_SCALER_MIN,
    vol_scaler_max=VOL_SCALER_MAX,
    use_ood_attenuation=USE_OOD_ATTENUATION,
    ood_threshold=OOD_THRESHOLD,
    ood_p90=ood_p90_s,
    ood_attenuation_alpha=OOD_ATTENUATION_ALPHA,
    use_ewma_corr=USE_EWMA_CORR,
    ewma_window=EWMA_WINDOW,
    use_autocorr_feature=USE_AUTOCORR_FEATURE,
    autocorr_window=AUTOCORR_WINDOW,
    use_disagreement_in_sigma=USE_DISAGREEMENT_IN_SIGMA,
    use_rolling_vol_in_sigma=USE_ROLLING_VOL_IN_SIGMA,
    rolling_vol_window=ROLLING_VOL_WINDOW,
)
print(f'  sigma_train_floor: {sigma_train_floor_s:.5f}')
if kelly_per_regime_s is not None:
    print(f'  kelly_per_regime: ' + ' '.join(f'{k:.2f}' for k in kelly_per_regime_s))

# 6. Quick PPO smoke train
t0 = time.time()
model_s = make_compact_ppo(
    hybrid_train_s, total_timesteps=20_000,
    learning_rate=PPO_LR, hidden=PPO_HIDDEN,
    kelly_fraction=PPO_KELLY_FRAC,
    kelly_per_regime=kelly_per_regime_s,
    n_regimes=HMM_STATES,
    regime_obs_offset=3 + (1 if USE_OOD_ATTENUATION else 0) + (1 if USE_EWMA_CORR else 0) + (1 if USE_AUTOCORR_FEATURE else 0),
    detach_kelly=DETACH_KELLY,
    verbose=0,
)
buf = io.StringIO()
with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
    model_s.learn(total_timesteps=20_000, progress_bar=False)
n_params = sum(p.numel() for p in model_s.policy.parameters())
print(f'  PPO 20k-step train: {time.time()-t0:5.1f}s  ({n_params:,} params)')

# 7. Eval
obs, _ = hybrid_test_s.reset(seed=42)
acts = []; done = False
while not done:
    a, _ = model_s.predict(obs, deterministic=True)
    acts.append(float(a[0]))
    obs, _, term, trunc, _ = hybrid_test_s.step(a)
    done = term or trunc
mets = hybrid_test_s.get_episode_metrics()
acts = np.array(acts)
print(f'\nSingle-split eval (split {TEST_SPLIT_IDX+1}):')
print(f'  Sharpe:   {mets.get("sharpe",0):+.3f}')
print(f'  Sortino:  {mets.get("sortino",0):+.3f}')
print(f'  MaxDD:    {mets.get("max_drawdown",0):+.3f}')
print(f'  Trades:   {mets.get("trade_count",0)}')
print(f'  |action| mean: {np.abs(acts).mean():.4f}')
print(f'  Kelly cap util: {model_s.policy.last_kelly:.3f}')

## Walk-forward main loop

33 expanding-window splits. Per split: train forecasters, HMM, ensemble, then PPO. Evaluate on the held-out test window. Saves a per-split checkpoint after every completed split (resilient to crashes).

In [ ]:
factory = WalkForwardEnvFactory(df, FEATURES, SPLITS, ENV_CONFIG)
all_results = []

print(f'Starting {len(SPLITS)}-split walk-forward (v3 hybrid)')
print('=' * 80)

for split_idx in range(len(SPLITS)):
    t_split = time.time()
    split = SPLITS[split_idx]
    
    base_train, base_test, _ = factory.get_split(split_idx)
    if base_train is None:
        print(f'[{split_idx+1:02d}] SKIP (insufficient data)')
        continue
    train_days = len(base_train.df)
    if train_days < MIN_TRAIN_DAYS:
        print(f'[{split_idx+1:02d}] SKIP ({train_days}d < {MIN_TRAIN_DAYS} min)')
        continue
    
    # 2. HMM first (so we have regime probs for regime-specialised forecaster training)
    t0 = time.time()
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
        hmm = RegimeHMM(n_states=HMM_STATES, vol_window=HMM_VOL_WINDOW, random_state=HMM_SEED).fit(base_train.df)
    train_rp_for_weights = hmm.predict_proba(base_train.df)
    regime_info = classify_regimes(hmm)
    t_hmm = time.time() - t0
    
    # 1. Forecasters (regime-specialised via per-sample weights)
    t0 = time.time()
    forecasters = build_default_forecasters(FEATURES, lookback=FORECASTER_LOOKBACK, epochs=FORECASTER_EPOCHS)
    # Assign each forecaster a primary regime for specialisation:
    primary_regime = {'lstm': 0, 'transformer': 0, 'xgboost': 1, 'cnn': 1, 'garch': 0}
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
        for fc in forecasters:
            if USE_REGIME_SPECIALISED_TRAINING:
                pri = primary_regime.get(fc.name, 0)
                pri = min(pri, HMM_STATES - 1)
                sw = train_rp_for_weights[:, pri] + 0.3   # +0.3 floor → no zero weight
                fc.fit(base_train.df, sample_weights=sw)
            else:
                fc.fit(base_train.df)
    t_forecast = time.time() - t0
    
    # 3. Stack predictions
    train_mus, train_sigs = stack_forecaster_predictions(forecasters, base_train.df)
    test_mus,  test_sigs  = stack_forecaster_predictions(forecasters, base_test.df)
    train_rp = hmm.predict_proba(base_train.df)
    test_rp  = hmm.predict_proba(base_test.df)
    
    # 4. Ensemble
    target_train = np.roll(base_train.df['log_return'].fillna(0).values, -1)
    ens = RegimeAwareEnsemble(
        n_regimes=HMM_STATES, n_forecasters=5,
        base_temp=ENSEMBLE_BASE_TEMP, alpha_temp=ENSEMBLE_ALPHA_TEMP,
    ).fit(train_mus, train_rp, target_train)
    
    # 5. v3.3: compute sigma_floor, regime classifier (hard Kelly caps),
    #         OOD p90 threshold from training disagreement
    sigma_train_floor = float(base_train.df['log_return'].std()) if USE_SIGMA_FLOOR else 0.0
    # v3.3: kelly_per_regime now uses hard regime-specific caps from classifier
    if USE_REGIME_KELLY:
        kelly_per_regime = regime_info['kelly_caps']
    else:
        kelly_per_regime = None
    target_vol        = TARGET_ANN_VOL if USE_VOL_TARGETING else None
    # v3.3 OOD p90 from training-set forecaster disagreement
    train_disagreement = train_mus.std(axis=1)
    ood_p90            = float(np.percentile(train_disagreement, 90)) if USE_OOD_ATTENUATION else None

    hybrid_train = HybridForecastEnv(
        base_train, train_mus, train_sigs, train_rp, ens,
        sigma_garch_idx=GARCH_FORECASTER_IDX,
        sigma_train_floor=sigma_train_floor,
        target_vol=target_vol,
        vol_scaler_min=VOL_SCALER_MIN,
        vol_scaler_max=VOL_SCALER_MAX,
        use_ood_attenuation=USE_OOD_ATTENUATION,
        ood_threshold=OOD_THRESHOLD,
        ood_p90=ood_p90,
        ood_attenuation_alpha=OOD_ATTENUATION_ALPHA,
        use_ewma_corr=USE_EWMA_CORR,
        ewma_window=EWMA_WINDOW,
        use_autocorr_feature=USE_AUTOCORR_FEATURE,
        autocorr_window=AUTOCORR_WINDOW,
        use_disagreement_in_sigma=USE_DISAGREEMENT_IN_SIGMA,
        use_rolling_vol_in_sigma=USE_ROLLING_VOL_IN_SIGMA,
        rolling_vol_window=ROLLING_VOL_WINDOW,
    )
    hybrid_test  = HybridForecastEnv(
        base_test, test_mus, test_sigs, test_rp, ens,
        sigma_garch_idx=GARCH_FORECASTER_IDX,
        sigma_train_floor=sigma_train_floor,
        target_vol=target_vol,
        vol_scaler_min=VOL_SCALER_MIN,
        vol_scaler_max=VOL_SCALER_MAX,
        use_ood_attenuation=USE_OOD_ATTENUATION,
        ood_threshold=OOD_THRESHOLD,
        ood_p90=ood_p90,
        ood_attenuation_alpha=OOD_ATTENUATION_ALPHA,
        use_ewma_corr=USE_EWMA_CORR,
        ewma_window=EWMA_WINDOW,
        use_autocorr_feature=USE_AUTOCORR_FEATURE,
        autocorr_window=AUTOCORR_WINDOW,
        use_disagreement_in_sigma=USE_DISAGREEMENT_IN_SIGMA,
        use_rolling_vol_in_sigma=USE_ROLLING_VOL_IN_SIGMA,
        rolling_vol_window=ROLLING_VOL_WINDOW,
    )
    
    timesteps_split = int(np.clip(train_days * 80, PPO_TIMESTEPS_BASE, 600_000))
    
    t0 = time.time()
    model = make_compact_ppo(
        hybrid_train,
        total_timesteps = timesteps_split,
        learning_rate   = PPO_LR,
        hidden          = PPO_HIDDEN,
        kelly_fraction  = PPO_KELLY_FRAC,
        kelly_per_regime= kelly_per_regime,
        n_regimes       = HMM_STATES,
        regime_obs_offset = 3 + (1 if USE_OOD_ATTENUATION else 0) + (1 if USE_EWMA_CORR else 0) + (1 if USE_AUTOCORR_FEATURE else 0),
        detach_kelly      = DETACH_KELLY,
        gamma           = PPO_GAMMA,
        gae_lambda      = PPO_GAE_LAMBDA,
        clip_range      = PPO_CLIP,
        ent_coef        = PPO_ENT_COEF,
        seed            = SEED,
        verbose         = 0,
    )
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
        model.learn(total_timesteps=timesteps_split, progress_bar=False)
    t_ppo = time.time() - t0
    
    # 6. Eval (5 episodes mean)
    all_mets = []
    for ep in range(EVAL_EPISODES):
        obs, _ = hybrid_test.reset(seed=SEED + ep)
        done = False
        while not done:
            a, _ = model.predict(obs, deterministic=True)
            obs, _, term, trunc, _ = hybrid_test.step(a)
            done = term or trunc
        all_mets.append(hybrid_test.get_episode_metrics())
    metrics = {k: float(np.mean([m.get(k, 0) for m in all_mets])) for k in all_mets[0]}
    calmar = metrics['ann_return'] / (abs(metrics['max_drawdown']) + 1e-10)
    
    # 7. Record
    W = ens.regime_weight_matrix().mean(axis=0)  # mean weight per forecaster across regimes
    result = {
        'split'        : split_idx + 1,
        'train_start'  : split['train_start'].date(),
        'test_start'   : split['test_start'].date(),
        'test_end'     : split['test_end'].date(),
        'sharpe'       : metrics['sharpe'],
        'sortino'      : metrics['sortino'],
        'calmar'       : calmar,
        'max_drawdown' : metrics['max_drawdown'],
        'ann_return'   : metrics['ann_return'],
        'win_rate'     : metrics['win_rate'],
        'trade_count'  : int(metrics['trade_count']),
        'w_lstm'       : float(W[0]),
        'w_transformer': float(W[1]),
        'w_xgboost'    : float(W[2]),
        'w_cnn'        : float(W[3]),
        'w_garch'      : float(W[4]),
        'train_days'   : train_days,
        'timesteps'    : timesteps_split,
        't_forecast_s' : round(t_forecast, 1),
        't_hmm_s'      : round(t_hmm, 1),
        't_ppo_s'      : round(t_ppo, 1),
        't_total_s'    : round(time.time() - t_split, 1),
    }
    all_results.append(result)
    
    print(
        f'[{split_idx+1:02d}/{len(SPLITS)}] '
        f'test {result["test_start"]} -> {result["test_end"]}  '
        f'Sharpe={result["sharpe"]:+.3f}  '
        f'Sortino={result["sortino"]:+.3f}  '
        f'MaxDD={result["max_drawdown"]:+.3f}  '
        f'trades={result["trade_count"]:3d}  '
        f'({result["t_total_s"]}s)'
    )
    
    # Per-split checkpoint
    pd.DataFrame(all_results).to_csv(
        os.path.join(RESULTS_DIR, 'v3_hybrid_results_checkpoint.csv'), index=False
    )

results_df = pd.DataFrame(all_results)
print('\n' + '=' * 80)
print(f'Mean Sharpe across {len(results_df)} splits: {results_df["sharpe"].mean():+.3f}')
print(f'Median Sharpe:                              {results_df["sharpe"].median():+.3f}')
print(f'Win rate (Sharpe > 0):                      {(results_df["sharpe"] > 0).mean():.0%}')

In [ ]:
# Per-split results
display_cols = ['split', 'test_start', 'test_end',
                'sharpe', 'sortino', 'calmar', 'max_drawdown',
                'ann_return', 'win_rate', 'trade_count']
print(results_df[display_cols].to_string(index=False, float_format='{:+.3f}'.format))

In [ ]:
# Comparison table vs prior models
print('\n--- Sharpe comparison across all v* models ---')
comparison = {'v3 Hybrid': results_df['sharpe'].values}
for label, fname in [
    ('Standard PPO', 'standard_ppo_sharpe.npy'),
    ('ARA-PPO v1',   'ara_ppo_v1_sharpe.npy'),
    ('ARA-PPO v2.2', 'v2.2_baseline_sharpe.npy'),
]:
    fp = os.path.join(RESULTS_DIR, fname)
    if os.path.exists(fp):
        comparison[label] = np.load(fp)
for label, sh in comparison.items():
    print(f'  {label:<15s}  mean={np.mean(sh):+.3f}  median={np.median(sh):+.3f}  '
          f'min={np.min(sh):+.3f}  max={np.max(sh):+.3f}  '
          f'pos={(np.array(sh) > 0).mean():.0%}')

# Save final results
np.save(os.path.join(RESULTS_DIR, 'v3_hybrid_sharpe.npy'), results_df['sharpe'].values)
results_df.to_csv(os.path.join(RESULTS_DIR, 'v3_hybrid_results.csv'), index=False)
print(f'\nSaved to {RESULTS_DIR}/v3_hybrid_*.{{csv,npy}}')

In [ ]:
# Plots: per-split Sharpe + ensemble weights + comparison
fig = plt.figure(figsize=(18, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

# 1. Per-split Sharpe
ax = fig.add_subplot(gs[0, 0])
colors = ['steelblue' if s > 0 else 'tomato' for s in results_df['sharpe']]
ax.bar(results_df['split'], results_df['sharpe'], color=colors)
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_title('Sharpe by Split (v3 Hybrid)')
ax.set_xlabel('Split'); ax.set_ylabel('Sharpe')

# 2. Ensemble weights distribution (mean across splits)
ax = fig.add_subplot(gs[0, 1])
weight_cols = ['w_lstm', 'w_transformer', 'w_xgboost', 'w_cnn', 'w_garch']
weights = results_df[weight_cols].mean()
ax.bar(['LSTM', 'Transformer', 'XGBoost', 'CNN', 'GARCH'], weights.values,
       color=['steelblue', 'orange', 'green', 'purple', 'tomato'])
ax.set_title('Mean Ensemble Weight per Forecaster')
ax.set_ylabel('Mean weight')

# 3. Sharpe distribution histogram
ax = fig.add_subplot(gs[0, 2])
ax.hist(results_df['sharpe'], bins=15, color='steelblue', edgecolor='black')
ax.axvline(0, color='k', lw=0.8, ls='--')
ax.axvline(results_df['sharpe'].mean(), color='red', lw=1.5, label=f'mean={results_df["sharpe"].mean():+.3f}')
ax.axvline(results_df['sharpe'].median(), color='green', lw=1.5, label=f'median={results_df["sharpe"].median():+.3f}')
ax.set_title('Sharpe Distribution')
ax.legend()

# 4. Cumulative Sharpe across splits
ax = fig.add_subplot(gs[1, 0])
ax.plot(results_df['split'], results_df['sharpe'].expanding().mean(),
        marker='o', color='steelblue')
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_title('Running mean Sharpe')
ax.set_xlabel('Split'); ax.set_ylabel('Cumulative mean Sharpe')

# 5. Comparison vs prior models (if files exist)
ax = fig.add_subplot(gs[1, 1])
labels, means = [], []
for label, sh in comparison.items():
    labels.append(label); means.append(np.mean(sh))
colors_cmp = ['steelblue' if m > 0 else 'tomato' for m in means]
ax.barh(labels, means, color=colors_cmp)
ax.axvline(0, color='k', lw=0.8, ls='--')
ax.set_title('Mean Sharpe vs prior models')

# 6. Trade count vs Sharpe scatter
ax = fig.add_subplot(gs[1, 2])
ax.scatter(results_df['trade_count'], results_df['sharpe'], color='steelblue')
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_xlabel('Trade count')
ax.set_ylabel('Sharpe')
ax.set_title('Trades vs Sharpe (per split)')

plt.suptitle('ARA-PPO v3 Hybrid Walk-Forward Results', fontsize=14, fontweight='bold')
plt.savefig(os.path.join(RESULTS_DIR, 'v3_hybrid_summary.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Per-split per-forecaster weights (heatmap)
weight_data = results_df[['w_lstm', 'w_transformer', 'w_xgboost', 'w_cnn', 'w_garch']].values.T
fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(weight_data, aspect='auto', cmap='viridis', vmin=0, vmax=1)
ax.set_yticks(range(5))
ax.set_yticklabels(['LSTM', 'Transformer', 'XGBoost', 'CNN', 'GARCH'])
ax.set_xticks(range(len(results_df)))
ax.set_xticklabels(results_df['split'].values, rotation=45)
ax.set_xlabel('Split')
ax.set_title('Forecaster weights (mean across regimes) by split')
plt.colorbar(im, ax=ax, label='Weight')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'v3_hybrid_weights_heatmap.png'), dpi=120, bbox_inches='tight')
plt.show()